# langchain-dkg — LangChain Memory Adapter for OriginTrail DKG v10

[![PyPI](https://img.shields.io/badge/pip-langchain--dkg-blue)](https://github.com/haroldboom/dkg-langchain)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)
[![Bounty](https://img.shields.io/badge/bounty-cfi--dkgv10--r1-orange)](https://docs.origintrail.io)

**Give any LangChain agent persistent, verifiable, queryable memory — every conversation turn stored as a cryptographically-linked Knowledge Asset on the Decentralized Knowledge Graph.**

This notebook demonstrates all three components of `langchain-dkg` using a **realistic offline mock** of the DKG v10 node API, so you can explore the package without needing to run a local node.

---

## What this package provides

| Class | LangChain base | What it does |
|---|---|---|
| `DKGChatMessageHistory` | `BaseChatMessageHistory` | Stores/retrieves conversation turns via DKG Working Memory |
| `DKGMemory` | `RunnableWithMessageHistory` factory | Wraps any LangChain Runnable with DKG-backed persistent memory |
| `DKGRetriever` | `BaseRetriever` | Executes SPARQL queries against the Knowledge Graph |
| `DKGClient` | — | Async HTTP client for the DKG v10 node API (port 9200) |

## Architecture

```
  Your LangChain Agent
         │
         ▼
  ┌──────────────────────────────────────────────────┐
  │              langchain-dkg                       │
  │                                                  │
  │  DKGMemory ──► DKGChatMessageHistory             │
  │                        │                         │
  │               DKGRetriever                       │
  │                        │                         │
  │                   DKGClient                      │
  └──────────────────────┬───────────────────────────┘
                         │  HTTP (port 9200)
                         ▼
             ┌─────────────────────┐
             │   DKG v10 Node      │
             │                     │
             │  Working Memory     │  ← private, local
             │  Shared W. Memory   │  ← gossip-replicated
             │  Verified Memory    │  ← on-chain, permanent
             └─────────────────────┘
```

Each stored message becomes a **tri-modal Knowledge Asset**: structural RDF triples + semantic embeddings + full-text index, all identified by a **UAL** (Universal Asset Locator).

## Setup

Install `langchain-dkg` from GitHub (replace with `pip install langchain-dkg` once published to PyPI).

In [ ]:
# Install from PyPI
!pip install langchain-dkg -q
!pip install langchain-core -q

print("Done.")

In [ ]:
import asyncio, hashlib, time, uuid
from langchain_dkg import DKGChatMessageHistory, DKGMemory, DKGRetriever, DKGClient
from langchain_core.messages import HumanMessage, AIMessage

print("langchain-dkg imported successfully.")

## Offline Mock — realistic DKG v10 API responses

`MockDKGClient` is a drop-in replacement for `DKGClient` that returns realistic API responses — same field names, same UAL format, simulated triple counts and embeddings — without needing a running DKG node.

To run against a **real** node instead, skip this cell and scroll to the *Live Mode* section at the bottom.

In [ ]:
class MockDKGClient:
    """Drop-in replacement for DKGClient — returns realistic DKG v10 API responses offline."""

    # Contract address used in all mock UALs
    _CONTRACT = "0x5cAC41237127F94C2D21dAE0B14BFeFa3BDcAAa"

    def __init__(self, base_url="http://localhost:9200", token="demo-token", **_):
        self._counter = 0
        self._store: list[dict] = []

    def _next_ual(self) -> str:
        self._counter += 1
        return f"did:dkg:otp:2043/{self._CONTRACT}/{self._counter:010d}"

    async def ping(self) -> bool:
        return True

    async def memory_turn(
        self, context_graph_id, markdown,
        session_uri=None, layer=None, sub_graph_name=None
    ) -> dict:
        ual = self._next_ual()
        n = self._counter
        file_hash = hashlib.sha256(markdown.encode()).hexdigest()
        embedding_id = f"emb-{uuid.uuid4().hex[:16]}"
        record = {
            "entityUri": ual,
            "label": markdown[:80],
            "snippet": markdown,
            "similarity": round(max(0.70, 0.97 - n * 0.02), 2),
            "memoryLayer": layer or "swm",
            "sourceFile": f"turn_{n:04d}.md",
            "sources": [ual],
        }
        self._store.append(record)
        return {
            "turnUri": ual,
            "fileHash": file_hash,
            "layer": layer or "swm",
            "graph": f"urn:graph:{context_graph_id}",
            "structuralTripleCount": 8 + (n % 6),
            "semanticTripleCount": 5 + (n % 4),
            "totalQuads": 13 + (n % 10),
            "embeddingId": embedding_id,
            "sessionUri": session_uri or f"urn:session:{uuid.uuid4().hex[:8]}",
        }

    async def memory_search(
        self, context_graph_id, query, limit=20, memory_layers=None
    ) -> dict:
        results = list(reversed(self._store))[:limit]
        return {
            "query": query,
            "contextGraphId": context_graph_id,
            "resultCount": len(results),
            "results": results,
        }

    async def query(
        self, sparql, paranet_id=None, graph_suffix=None, include_workspace=True
    ) -> dict:
        C = self._CONTRACT
        return {
            "results": {
                "bindings": [
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000001"},
                     "predicate": {"value": "http://schema.org/name"},
                     "object": {"value": "OriginTrail"}},
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000001"},
                     "predicate": {"value": "http://schema.org/description"},
                     "object": {"value": "A decentralized knowledge graph protocol enabling trusted AI with verifiable data provenance."}},
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000002"},
                     "predicate": {"value": "http://schema.org/name"},
                     "object": {"value": "Knowledge Asset"}},
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000002"},
                     "predicate": {"value": "http://schema.org/description"},
                     "object": {"value": "An ownable, cryptographically-linked container of structured knowledge on the DKG."}},
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000003"},
                     "predicate": {"value": "http://schema.org/name"},
                     "object": {"value": "Working Memory"}},
                    {"subject": {"value": f"did:dkg:otp:2043/{C}/0000000003"},
                     "predicate": {"value": "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"},
                     "object": {"value": "http://schema.org/MemoryLayer"}},
                ]
            }
        }


mock = MockDKGClient()
print("MockDKGClient ready — simulating DKG v10 node responses offline.")

---
## Demo 1 — DKGChatMessageHistory

Store and retrieve conversation turns as **Knowledge Assets** on the DKG.

Each call to `aadd_message()` maps to one `POST /api/memory/turn` request. The DKG node processes the message as Markdown, generates structural RDF triples, semantic embeddings, and a full-text index — then returns a **UAL** (Universal Asset Locator) that cryptographically identifies the stored knowledge.

`aget_messages()` calls `POST /api/memory/search` — **tri-modal retrieval** across all three stores.

In [ ]:
# 1a — Store conversation turns

history = DKGChatMessageHistory(
    context_graph_id="demo-colab",
    client=mock,
    layer="wm",  # Working Memory — private, stays on your node
    session_uri="urn:session:colab-demo-001",
)

turns = [
    HumanMessage(content="What is OriginTrail?"),
    AIMessage(content="OriginTrail is a decentralized knowledge graph protocol that enables trusted AI with verifiable data provenance."),
    HumanMessage(content="What is a Knowledge Asset?"),
    AIMessage(content="A Knowledge Asset is an ownable, cryptographically-linked container of structured knowledge, uniquely identified by a UAL."),
    HumanMessage(content="What memory layers does DKG v10 support?"),
    AIMessage(content="DKG v10 has three layers: Working Memory (private), Shared Working Memory (gossip-replicated), and Verified Memory (on-chain, permanent)."),
]

print(f"Storing {len(turns)} conversation turns in DKG Working Memory...\n")

async def store_turns():
    for msg in turns:
        await history.aadd_message(msg)
        role = "Human" if isinstance(msg, HumanMessage) else "AI   "
        last = mock._store[-1]
        print(f"  [{role}] {msg.content[:60]}...")
        print(f"    UAL   : {last['entityUri']}")
        print(f"    Layer : {last['memoryLayer']}  |  Source: {last['sourceFile']}")
        print()

await store_turns()

In [ ]:
# 1b — Semantic search / retrieval

search_history = DKGChatMessageHistory(
    context_graph_id="demo-colab",
    client=mock,
    search_query="Knowledge Asset",
    search_limit=4,
)

print('Tri-modal search: "Knowledge Asset"\n')

async def search_turns():
    result = await mock.memory_search(
        context_graph_id="demo-colab",
        query="Knowledge Asset",
        limit=4,
    )
    print(f"  {result['resultCount']} turns retrieved:\n")
    for item in result["results"]:
        snippet = item.get("snippet", item.get("label", ""))
        print(f"  [{item['similarity']:.2f}] {snippet[:70]}...")
        print(f"         UAL   : {item['entityUri']}")
        print(f"         Layer : {item['memoryLayer']}")
        print()

await search_turns()

---
## Demo 2 — DKGMemory + LangChain LCEL Chain

`DKGMemory.wrap_chain()` wraps any LangChain `Runnable` with `RunnableWithMessageHistory`, using `DKGChatMessageHistory` as the history backend.

The agent automatically:
- **reads** relevant history from DKG before each turn (semantic search)
- **writes** both the human turn and AI response back to DKG after each turn

This demo uses `FakeListChatModel` (no API key required) — swap in `ChatOpenAI` or any LangChain-compatible LLM.

In [ ]:
import os
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

OPENAI_KEY = os.environ.get("OPENAI_API_KEY", "")

if OPENAI_KEY:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_label = "gpt-4o-mini"
else:
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    llm = FakeListChatModel(responses=[
        "DKG v10 has three memory layers: Working Memory (local and private), "
        "Shared Working Memory (gossip-replicated across trusted peers), and "
        "Verified Memory (anchored on-chain — permanent and trustless).",
        "For sensitive private data, use Working Memory (layer='wm'). "
        "It never leaves your local node and is never gossiped or published.",
    ])
    llm_label = "FakeListChatModel (set OPENAI_API_KEY to use a real LLM)"

print(f"LLM            : {llm_label}")
print(f"Memory backend : DKG v10 Working Memory  (context_graph_id='demo-colab')")
print(f"Session        : demo-session-01\n")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to a Decentralized Knowledge Graph."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# DKGMemory.wrap_chain() — one call to bind any chain to DKG persistent memory
chain_with_memory = DKGMemory.wrap_chain(
    prompt | llm,
    context_graph_id="demo-colab",
    client=mock,
    search_limit=4,
    history_messages_key="history",
)

print("Chain assembled: ChatPromptTemplate | LLM | DKGMemory\n")

questions = [
    "Summarise the DKG v10 memory layer architecture.",
    "Which layer is best for sensitive private data?",
]

for q in questions:
    print(f"Human: {q}")
    response = chain_with_memory.invoke(
        {"input": q},
        config={"configurable": {"session_id": "demo-session-01"}},
    )
    answer = response.content if hasattr(response, "content") else str(response)
    print(f"AI:    {answer}")
    print()

print(f"Turns now in DKG Working Memory: {len(mock._store)} Knowledge Assets")

---
## Demo 3 — DKGRetriever (SPARQL)

`DKGRetriever` implements LangChain's `BaseRetriever` interface. It executes SPARQL SELECT queries against the DKG node and returns each result triple as a `Document` with provenance metadata.

This makes the Knowledge Graph a first-class RAG source — plug it directly into `RetrievalQA`, `ConversationalRetrievalChain`, or any other LangChain retrieval chain.

In [ ]:
retriever = DKGRetriever(client=mock, limit=6, include_workspace=True)

print("DKGRetriever — SPARQL query results as LangChain Documents\n")

for query in ["OriginTrail", "Knowledge Asset"]:
    print(f'Query: "{query}"')
    docs = await retriever._aget_relevant_documents(query)
    print(f"  {len(docs)} triples returned as Documents:\n")
    for doc in docs:
        print(f"  page_content : {doc.page_content}")
        print(f"  metadata     : {doc.metadata}")
        print()
    print()

---
## DKG v10 Memory Layers — reference

| Layer | API param | Scope | Cost | Use case |
|---|---|---|---|---|
| **Working Memory** | `layer="wm"` | Local node only | Free | Private agent memory, sensitive data |
| **Shared Working Memory** | `layer="swm"` (default) | Gossip-replicated across peers | Free | Collaborative agents, shared context |
| **Verified Memory** | (via `shared_memory_publish`) | On-chain, permanent | TRAC tokens | Auditable records, public knowledge assets |

`langchain-dkg` writes to `wm` or `swm` by default. Promoting to Verified Memory (on-chain) is an explicit, Curator-authorized operation — it is never triggered automatically.

---
## Live Mode — run against a real DKG v10 node

Replace `client=mock` with a real `DKGClient` anywhere in this notebook.

In [ ]:
# ── Live mode setup ──────────────────────────────────────────────────────────
#
# 1. Install and start a DKG v10 node:
#      npm install -g @origintrail-official/dkg
#      dkg init && dkg start
#
# 2. Get your bearer token:
#      export DKG_TOKEN=$(dkg auth show)
#
# 3. Uncomment the lines below and re-run the demo cells with `client=live_client`

import os

DKG_TOKEN = os.environ.get("DKG_TOKEN", "")

if DKG_TOKEN:
    live_client = DKGClient(token=DKG_TOKEN)
    connected = await live_client.ping()
    if connected:
        print("Connected to DKG v10 node at http://localhost:9200")
        print("Replace client=mock with client=live_client in any cell above.")
    else:
        print("Node not reachable — run: dkg start && sleep 20")
else:
    print("DKG_TOKEN not set. Running in mock (offline) mode.")
    print("Set it with:  import os; os.environ['DKG_TOKEN'] = '<your token>'")

---
## Links

- **GitHub**: https://github.com/haroldboom/dkg-langchain
- **PyPI**: `pip install langchain-dkg`
- **OriginTrail DKG v10 docs**: https://docs.origintrail.io
- **Bounty programme**: https://docs.origintrail.io/origintrail-v9-v10/origintrail-dkg-v10-bounty-program
- **Submission tag**: `cfi-dkgv10-r1`